# Tarea de clasificación de texto

Como mencionamos, nos enfocaremos en una tarea sencilla de clasificación de texto basada en el dataset **AG_NEWS**, que consiste en clasificar titulares de noticias en una de 4 categorías: World (Mundo), Sports (Deportes), Business (Negocios) y Sci/Tech (Ciencia/Tecnología).

## El dataset

Este dataset viene incluido en el módulo [`torchtext`](https://github.com/pytorch/text), por lo que podemos acceder a él fácilmente.

In [1]:
import torch
import torchtext
import os
import collections
os.makedirs('./data',exist_ok=True)
train_dataset, test_dataset = torchtext.datasets.AG_NEWS(root='./data')
classes = ['World', 'Sports', 'Business', 'Sci/Tech']

Aquí, `train_dataset` y `test_dataset` contienen colecciones que devuelven pares de etiqueta (número de la clase) y texto, respectivamente; por ejemplo:

In [2]:
list(train_dataset)[0]

(3,
 "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.")

Ahora imprimamos los primeros 5 titulares de nuestro dataset: 

In [5]:
for i,x in zip(range(5),train_dataset):
    print(f"**{classes[x[0]]}** -> {x[1]}")


**Sci/Tech** -> Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
**Sci/Tech** -> Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\which has a reputation for making well-timed and occasionally\controversial plays in the defense industry, has quietly placed\its bets on another part of the market.
**Sci/Tech** -> Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\about the economy and the outlook for earnings are expected to\hang over the stock market next week during the depth of the\summer doldrums.
**Sci/Tech** -> Iraq Halts Oil Exports from Main Southern Pipeline (Reuters) Reuters - Authorities have halted oil export\flows from the main pipeline in southern Iraq after\intelligence showed a rebel militia could strike\infrastructure, an oil official said on Saturday.
**Sci/Tech** -> Oil prices soar to

Como los datasets son iteradores, si queremos usar los datos varias veces debemos convertirlos en una lista:

In [3]:
train_dataset, test_dataset = torchtext.datasets.AG_NEWS(root='./data')
train_dataset = list(train_dataset)
test_dataset = list(test_dataset)

## Tokenización

Ahora necesitamos convertir el texto en **números** que puedan representarse como tensores. Si queremos una representación a nivel de palabras, debemos hacer dos cosas:
* usar un **tokenizador** para dividir el texto en **tokens**
* construir un **vocabulario** con esos tokens.

In [4]:
tokenizer = torchtext.data.utils.get_tokenizer('basic_english')
tokenizer('He said: hello')

['he', 'said', 'hello']

In [5]:
counter = collections.Counter()
for (label, line) in train_dataset:
    counter.update(tokenizer(line))
vocab = torchtext.vocab.vocab(counter, min_freq=1)

Usando el vocabulario, podemos codificar fácilmente nuestro string tokenizado como una secuencia de números:

In [19]:
vocab_size = len(vocab)
print(f"Vocab size if {vocab_size}")

stoi = vocab.get_stoi() # diccionario para convertir tokens en índices

def encode(x):
    return [stoi[s] for s in tokenizer(x)]

encode('I love to play with my words')

Vocab size if 95810


[599, 3279, 97, 1220, 329, 225, 7368]

## Representación de texto Bag of Words

Como las palabras representan significado, a veces podemos deducir el significado de un texto con solo mirar las palabras individuales, sin importar su orden en la oración. Por ejemplo, al clasificar noticias, palabras como *weather* (clima) o *snow* (nieve) probablemente indiquen un *pronóstico del tiempo*, mientras que palabras como *stocks* (acciones) o *dollar* (dólar) apuntarían a *noticias financieras*.

La representación vectorial **Bag of Words** (BoW, "bolsa de palabras") es la representación vectorial tradicional más utilizada. Cada palabra se asocia a un índice del vector, y cada elemento del vector contiene el número de apariciones de esa palabra en un documento dado.

![Imagen que muestra cómo se representa en memoria un vector bag of words.](images/bag-of-words-example.png)

> **Nota**: También se puede pensar en BoW como la suma de todos los vectores one-hot de las palabras individuales del texto.

A continuación se muestra un ejemplo de cómo generar una representación bag of words usando la biblioteca de Python Scikit Learn:

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
corpus = [
        'I like hot dogs.',
        'The dog ran fast.',
        'Its hot outside.',
    ]
vectorizer.fit_transform(corpus)
vectorizer.transform(['My dog likes hot dogs on a hot day.']).toarray()

array([[1, 1, 0, 2, 0, 0, 0, 0, 0]], dtype=int64)

Para calcular el vector bag-of-words a partir de la representación vectorial de nuestro dataset AG_NEWS, podemos usar la siguiente función:

In [20]:
vocab_size = len(vocab)

def to_bow(text,bow_vocab_size=vocab_size):
    res = torch.zeros(bow_vocab_size,dtype=torch.float32)
    for i in encode(text):
        if i<bow_vocab_size:
            res[i] += 1
    return res

print(to_bow(train_dataset[0][1]))

tensor([2., 1., 2.,  ..., 0., 0., 0.])


> **Nota:** Aquí usamos la variable global `vocab_size` para especificar el tamaño por defecto del vocabulario. Como el vocabulario suele ser bastante grande, podemos limitar su tamaño a las palabras más frecuentes. Prueba a reducir el valor de `vocab_size`, ejecuta el código siguiente y observa cómo afecta a la exactitud (accuracy). Deberías esperar cierta caída en la exactitud, aunque no dramática, a cambio de un mejor rendimiento.

## Entrenamiento de un clasificador BoW

Ahora que aprendimos a construir la representación Bag-of-Words de nuestro texto, entrenemos un clasificador sobre ella. Primero, debemos convertir nuestro dataset para el entrenamiento de modo que todas las representaciones vectoriales posicionales se transformen en representaciones bag-of-words. Esto se logra pasando la función `bowify` como parámetro `collate_fn` al `DataLoader` estándar de torch:

In [21]:
from torch.utils.data import DataLoader
import numpy as np 

# esta función collate recibe una lista de batch_size tuplas y debe 
# devolver un par de tensores etiqueta-características para todo el minibatch
def bowify(b):
    return (
            torch.LongTensor([t[0]-1 for t in b]),
            torch.stack([to_bow(t[1]) for t in b])
    )

train_loader = DataLoader(train_dataset, batch_size=16, collate_fn=bowify, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, collate_fn=bowify, shuffle=True)

Ahora definamos una red neuronal clasificadora simple que contiene una sola capa lineal. El tamaño del vector de entrada es igual a `vocab_size`, y el tamaño de salida corresponde al número de clases (4). Como estamos resolviendo una tarea de clasificación, la función de activación final es `LogSoftmax()`.

In [22]:
net = torch.nn.Sequential(torch.nn.Linear(vocab_size,4),torch.nn.LogSoftmax(dim=1))

Ahora definiremos el ciclo de entrenamiento estándar de PyTorch. Como nuestro dataset es bastante grande, con fines didácticos entrenaremos solo una época, y a veces incluso menos de una época (el parámetro `epoch_size` nos permite limitar el entrenamiento). También reportaremos la exactitud de entrenamiento acumulada durante el entrenamiento; la frecuencia del reporte se especifica con el parámetro `report_freq`.

In [24]:
def train_epoch(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.NLLLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,features in dataloader:
        optimizer.zero_grad()
        out = net(features)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count

In [25]:
train_epoch(net,train_loader,epoch_size=15000)

3200: acc=0.8028125
6400: acc=0.8371875
9600: acc=0.8534375
12800: acc=0.85765625


(0.026090790722161722, 0.8620069296375267)

## Bigramas, trigramas y N-gramas

Una limitación del enfoque bag of words es que algunas palabras forman parte de expresiones de varias palabras; por ejemplo, 'hot dog' tiene un significado completamente distinto al de las palabras 'hot' (caliente) y 'dog' (perro) en otros contextos. Si representamos las palabras 'hot' y 'dog' siempre con los mismos vectores, podemos confundir a nuestro modelo.

Para abordar esto, en los métodos de clasificación de documentos se usan a menudo **representaciones de N-gramas**, donde la frecuencia de cada palabra, par de palabras o trío de palabras es una característica útil para entrenar clasificadores. En la representación de bigramas, por ejemplo, agregaremos al vocabulario todos los pares de palabras, además de las palabras originales.

A continuación se muestra un ejemplo de cómo generar una representación bag of words de bigramas usando Scikit Learn:


In [26]:
bigram_vectorizer = CountVectorizer(ngram_range=(1, 2), token_pattern=r'\b\w+\b', min_df=1)
corpus = [
        'I like hot dogs.',
        'The dog ran fast.',
        'Its hot outside.',
    ]
bigram_vectorizer.fit_transform(corpus)
print("Vocabulary:\n",bigram_vectorizer.vocabulary_)
bigram_vectorizer.transform(['My dog likes hot dogs on a hot day.']).toarray()


Vocabulary:
 {'i': 7, 'like': 11, 'hot': 4, 'dogs': 2, 'i like': 8, 'like hot': 12, 'hot dogs': 5, 'the': 16, 'dog': 0, 'ran': 14, 'fast': 3, 'the dog': 17, 'dog ran': 1, 'ran fast': 15, 'its': 9, 'outside': 13, 'its hot': 10, 'hot outside': 6}


array([[1, 0, 1, 0, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]],
      dtype=int64)

La principal desventaja del enfoque de N-gramas es que el tamaño del vocabulario crece extremadamente rápido. En la práctica, necesitamos combinar la representación de N-gramas con alguna técnica de reducción de dimensionalidad, como los *embeddings*, que veremos en la próxima unidad.

Para usar la representación de N-gramas en nuestro dataset **AG News**, necesitamos construir un vocabulario especial de n-gramas:

In [27]:
counter = collections.Counter()
for (label, line) in train_dataset:
    l = tokenizer(line)
    counter.update(torchtext.data.utils.ngrams_iterator(l,ngrams=2))
    
bi_vocab = torchtext.vocab.vocab(counter, min_freq=1)

print("Bigram vocabulary length = ",len(bi_vocab))

Bigram vocabulary length =  1308842


Luego podríamos usar el mismo código anterior para entrenar el clasificador; sin embargo, sería muy ineficiente en memoria. En la próxima unidad entrenaremos un clasificador de bigramas usando embeddings.

> **Nota:** Puedes conservar solo los n-gramas que aparecen en el texto más de un número determinado de veces. Esto asegura que los bigramas poco frecuentes se omitan y reduce significativamente la dimensionalidad. Para hacerlo, asigna un valor más alto al parámetro `min_freq` y observa cómo cambia el tamaño del vocabulario.

## Frecuencia de término – frecuencia inversa de documento (TF-IDF)

En la representación BoW, las apariciones de las palabras se ponderan por igual, sin importar la palabra en sí. Sin embargo, es evidente que las palabras frecuentes, como *a*, *in*, etc., son mucho menos importantes para la clasificación que los términos especializados. De hecho, en la mayoría de las tareas de NLP algunas palabras son más relevantes que otras.

**TF-IDF** significa **term frequency–inverse document frequency** (frecuencia de término – frecuencia inversa de documento). Es una variante de bag of words en la que, en lugar de un valor binario 0/1 que indica la aparición de una palabra en un documento, se usa un valor de punto flotante relacionado con la frecuencia de aparición de la palabra en el corpus.

Más formalmente, el peso $w_{ij}$ de una palabra $i$ en el documento $j$ se define como:
$$
w_{ij} = tf_{ij}\times\log({N\over df_i})
$$
donde
* $tf_{ij}$ es el número de apariciones de $i$ en $j$, es decir, el valor BoW que vimos antes
* $N$ es el número de documentos de la colección
* $df_i$ es el número de documentos de toda la colección que contienen la palabra $i$

El valor TF-IDF $w_{ij}$ aumenta proporcionalmente al número de veces que una palabra aparece en un documento y se compensa con el número de documentos del corpus que contienen la palabra, lo que ayuda a ajustar por el hecho de que algunas palabras aparecen con más frecuencia que otras. Por ejemplo, si la palabra aparece en *todos* los documentos de la colección, $df_i=N$ y $w_{ij}=0$, y esos términos se ignorarían por completo.

Se puede crear fácilmente una vectorización TF-IDF de un texto usando Scikit Learn:

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(ngram_range=(1,2))
vectorizer.fit_transform(corpus)
vectorizer.transform(['My dog likes hot dogs on a hot day.']).toarray()

array([[0.43381609, 0.        , 0.43381609, 0.        , 0.65985664,
        0.43381609, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        ]])

## Conclusión

Aunque las representaciones TF-IDF asignan un peso de frecuencia a las distintas palabras, no son capaces de representar el significado ni el orden. Como dijo el famoso lingüista J. R. Firth en 1935: “El significado completo de una palabra es siempre contextual, y ningún estudio del significado al margen del contexto puede tomarse en serio.” Más adelante en el curso aprenderemos a capturar información contextual del texto mediante el modelado de lenguaje.
